In [ ]:
%pip install sodapy prefect python-dotenv sqlalchemy streamlit psycopg2-binary

In [19]:
# extract.py
import pandas as pd
from sodapy import Socrata
from dotenv import load_dotenv
import os
from prefect import task

@task(retries=3, retry_delay_seconds=[10, 10, 10])
def extract_data():
    '''
    Extract incident data reported to the Cambridge Police Department
    using the Socrata Open Data API.
    Return the incident data as a Pandas DataFrame.
    '''
    load_dotenv()
    APP_TOKEN = os.getenv("SOCRATA_APP_TOKEN")

    client = Socrata(
        "data.cambridgema.gov",
        APP_TOKEN,
        timeout=30
    )

    DATASET_ID = "3gki-wyrb"
    results = client.get_all(DATASET_ID)

    results_df = pd.DataFrame.from_records(results)
    return results_df

In [20]:
# validate.py
import pandas as pd
from collections import Counter
from prefect import task

SCHEMA_COLS = [
    "date_time",
    "id",
    "type",
    "subtype",
    "location",
    "last_updated",
    "description"
]

def check_schema(df):
    # check if all expected columns are present
    if Counter(df.columns) != Counter(SCHEMA_COLS):
        raise ValueError("Schema does not match with the expected schema.")

def check_numeric_id(df):
    # convert id to numeric, non-numeric values become NaN
    df["id"] = pd.to_numeric(df["id"], errors="coerce")
    return df

def check_datetime_format(df):
    # validate datetimes follow ISO 8601 format
    try:
        pd.to_datetime(df["date_time"], format="ISO8601", errors="raise")
    except Exception:
        raise ValueError("Datetime does not follow ISO 8601 format.")

def check_missing_values(df):
    # check no missing values in required columns
    required_cols = ["date_time", "id", "type", "location"]
    if df[required_cols].isnull().any().any():
        df.dropna(subset=required_cols, inplace=True)

@task
def validate_data(df):
    df = df.copy()
    check_schema(df)
    df = check_numeric_id(df)
    check_missing_values(df)
    check_datetime_format(df)
    return df

In [21]:
# transform.py
import pandas as pd
from prefect import task

@task
def transform_data(df):
    df = df.copy()

    # remove rows with NaN id and duplicates
    df.dropna(subset=["id"], inplace=True)
    df.drop_duplicates(subset=["id"], inplace=True)
    df["id"] = df["id"].astype(int)

    # convert to datetime
    df["date_time"] = pd.to_datetime(df["date_time"])
    df["last_updated"] = pd.to_datetime(df["last_updated"])

    # extract year/month/day/time
    df["year"] = df["date_time"].dt.year
    df["month"] = df["date_time"].dt.month
    df["day"] = df["date_time"].dt.day
    df["hour"] = df["date_time"].dt.hour
    df["minute"] = df["date_time"].dt.minute
    df["second"] = df["date_time"].dt.second

    return df

In [ ]:
# load.py
import os
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text
from prefect import task

@task
def load_data(df):
    load_dotenv()

    db_url = os.getenv("DATABASE_URL")
    if not db_url:
        db_url = URL.create(
            "postgresql+psycopg2",
            username=os.getenv("POSTGRES_USER", "cpd_user"),
            password=os.getenv("POSTGRES_PASSWORD", "cpd_pass"),
            host=os.getenv("POSTGRES_HOST", "localhost"),
            port=int(os.getenv("POSTGRES_PORT", "5432")),
            database=os.getenv("POSTGRES_DB", "cpd_db"),
        )

    engine = create_engine(db_url)

    create_table_query = """
    CREATE TABLE IF NOT EXISTS cpd_incidents (
        date_time TIMESTAMP,
        id INTEGER PRIMARY KEY,
        type TEXT,
        subtype TEXT,
        location TEXT,
        description TEXT,
        last_updated TIMESTAMP,
        year INTEGER,
        month INTEGER,
        day INTEGER,
        hour INTEGER,
        minute INTEGER,
        second INTEGER
    );
    """

    with engine.begin() as conn:
        conn.execute(text(create_table_query))

    df.to_sql("cpd_incidents", con=engine, if_exists="replace", index=False)
    print("Data loaded successfully. Total rows:", len(df))

In [ ]:
from html import escape
import os
from IPython.display import HTML, display

airflow_url = os.getenv("AIRFLOW_URL", "http://localhost:8080")
display(HTML(
    "<p>Trigger <code>cpd_crime_trends_etl</code> in Airflow, wait for the DAG "
    "to succeed, then start the dashboard: "
    f'<a href="{escape(airflow_url, quote=True)}">{escape(airflow_url)}</a></p>'
))

In [ ]:
%%writefile dashboard.py
import os

import pandas as pd
import streamlit as st
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text
from sqlalchemy.exc import SQLAlchemyError

st.title("Cambridge Crime Trends Dashboard")
st.write("ETL Pipeline Dashboard - Cambridge Police Department Daily Log")

load_dotenv()
database_url = os.getenv("DATABASE_URL")
if database_url:
    engine = create_engine(database_url)
else:
    engine = create_engine(
        URL.create(
            "postgresql+psycopg2",
            username=os.getenv("POSTGRES_USER", "cpd_user"),
            password=os.getenv("POSTGRES_PASSWORD", "cpd_pass"),
            host=os.getenv("POSTGRES_HOST", "localhost"),
            port=int(os.getenv("POSTGRES_PORT", "5432")),
            database=os.getenv("POSTGRES_DB", "cpd_db"),
        )
    )

try:
    with engine.connect() as connection:
        df = pd.read_sql(text("SELECT * FROM cpd_incidents"), con=connection)
except SQLAlchemyError as exc:
    st.error(
        "Could not read cpd_incidents from PostgreSQL. Start the Docker Compose "
        "services, run the Airflow DAG, and check DATABASE_URL/.env settings. "
        f"Database error: {exc}"
    )
    st.stop()

if df.empty:
    st.info("No incident rows are available yet. Trigger the Airflow DAG and reload.")
    st.stop()

# simple filter by year
years = sorted(df["year"].dropna().astype(int).unique().tolist())
if not years:
    st.info("No incidents with a valid year are available yet.")
    st.stop()

selected_year = st.selectbox("Select Year", years, index=len(years)-1)
filtered_df = df[df["year"] == selected_year]

# summary counts
st.subheader("Summary")
col1, col2, col3 = st.columns(3)
col1.metric("Total Incidents", len(filtered_df))
col2.metric("Crime Types", filtered_df["type"].nunique())
col3.metric("Unique Streets", filtered_df["location"].nunique())

# chart 1: top 10 crime types
st.subheader("Top 10 Incident Types")
top_types = filtered_df["type"].value_counts().head(10)
st.bar_chart(top_types)

# chart 2: incidents by hour of day
st.subheader("Incidents by Hour of Day")
hourly_counts = filtered_df["hour"].value_counts().sort_index()
st.line_chart(hourly_counts)

# chart 3: incidents by month
st.subheader("Incidents by Month")
monthly_counts = filtered_df["month"].value_counts().sort_index()
st.bar_chart(monthly_counts)

# show table data
st.subheader("Recent Police Log Entries")
st.dataframe(filtered_df.head(100))


In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path
from IPython.display import HTML, display

port = int(os.getenv("STREAMLIT_SERVER_PORT", "8501"))
dashboard_path = Path.cwd() / "dashboard.py"
if not dashboard_path.exists():
    raise FileNotFoundError(f"Dashboard file not found: {dashboard_path}")

dashboard_process = globals().get("dashboard_process")
if dashboard_process is None or dashboard_process.poll() is not None:
    dashboard_process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            str(dashboard_path),
            "--server.address",
            "0.0.0.0",
            "--server.port",
            str(port),
            "--server.headless",
            "true",
        ],
        cwd=str(dashboard_path.parent),
        env=os.environ.copy(),
    )
    time.sleep(2)
    if dashboard_process.poll() is not None:
        raise RuntimeError(
            "Streamlit exited during startup. Check the notebook kernel terminal "
            "for the error and confirm Streamlit is installed."
        )

dashboard_url = os.getenv("DASHBOARD_URL", f"http://localhost:{port}")
display(HTML(
    f'<p>Dashboard is starting. If this is a remote VS Code workspace, forward '
    f'port {port} in the Ports panel. '
    f'<a href="{dashboard_url}" target="_blank">Open dashboard</a></p>'
))